# Imported DOT to CSCL mapper

This is the older mapping notebook from `GITHUB_NEWER`. It is kept for reference.


In [2]:
import pandas as pd

In [3]:
dot = pd.read_csv("dot_inhouse_resurfacing.csv")

/var/folders/zh/4snkqr657fb06hs2r74_4xpm0000gn/T/ipykernel_16550/528813035.py:1: DtypeWarning: Columns (0,5,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  dot = pd.read_csv("dot_inhouse_resurfacing.csv")


In [35]:
dot = dot[["OFT Code", "Borough Code", "Location On Street"]]

In [36]:
dot.shape

(598060, 3)

In [66]:
cscl = pd.read_csv("CSCL.csv")

In [67]:
cscl[cscl["PHYSICALID"] == 140711]

,the_geom,PHYSICALID,L_LOW_HN,L_HIGH_HN,R_LOW_HN,R_HIGH_HN,L_ZIP,R_ZIP,STATUS,BIKE_LANE,...,Post Directional,Post Modifier,Full Street Name,BIKE TRAFFIC DIRECTION,SHAPE__Length,GlobalID,SEGMENT_TYPE,SEGMENT_TYPE_VALUE,STREET NAME,Street Name Label
1550,MULTILINESTRING ((-73.98115480613 40.732979088...,140711,0,0,282,298,10009.0,10009.0,2,NaN,...,NaN,NaN,1 AVE,NaN,62.22621,64e148e5-01a7-4398-928f-11953c3cb5e9,NaN,NaN,1,1 AVE


In [34]:
cscl = cscl[["Borough Code", "Full Street Name", "STREET NAME", "Street Name Label", "PHYSICALID"]]

In [16]:
cscl.head()

,Borough Code,Full Street Name,STREET NAME,Street Name Label,PHYSICALID
0,3,AVE N,N,AVE N,46810
1,2,HONE AVE,HONE,HONE AVE,86757
2,4,48 ST,48,48 ST,84282
3,1,LAIGHT ST,LAIGHT,LAIGHT ST,79741
4,1,W 60 ST,60,W 60 ST,191409


In [37]:
# Borough code used in DOT df mapping to CSCL borough code

borough_dct = {'M':1, 'X':2, 'B':3, 'Q':4, 'S':5}

In [166]:
import pandas as pd
import re

# --- Borough code mapping ---
borough_dct = {'M': '1', 'X': '2', 'B': '3', 'Q': '4', 'S': '5'}

# --- Abbreviation and ordinal tables ---
ABBREV = {
    'st': 'street', 'st.': 'street', 'street.': 'street',
    'ave': 'avenue', 'av': 'avenue', 'av.': 'avenue',
    'rd': 'road', 'rd.': 'road',
    'blvd': 'boulevard', 'blvd.': 'boulevard',
    'pl': 'place', 'plz': 'plaza', 'pl.': 'place',
    'ct': 'court', 'ctr': 'center',
    'ln': 'lane', 'dr': 'drive', 'ter': 'terrace',
    'hwy': 'highway', 'pkwy': 'parkway',
    'sq': 'square', 'e': 'east', 'e.': 'east', 'w': 'west', 'w.': 'west',
    's': 'south', 's.': 'south', 'n': 'north', 'n.': 'north',
    'wash': 'washington', 'wash.': 'washington', 'ft': 'fort',
    'pl': 'place', 'pl.': 'place', 'aly': 'alley', 'aly.': 'alley',
    'cres': 'crescent', 'cres.': 'crescent', 'cr': 'crescent', 'cr.': 'crescent',
    'cir': 'circle', 'cir.': 'circle', 'grn': 'green', 'grn.': 'green',
    'hl': 'hill', 'hl.': 'hill', 'mt': 'mount', 'mt.': 'mount'
}

ORDINAL = {
    'first': '1', '1st': '1',
    'second': '2', '2nd': '2',
    'third': '3', '3rd': '3',
    'fourth': '4', '4th': '4',
    'fifth': '5', '5th': '5',
    'sixth': '6', '6th': '6',
    'seventh': '7', '7th': '7',
    'eighth': '8', '8th': '8',
    'ninth': '9', '9th': '9',
    'tenth': '10', '10th': '10'
}

# --- Alias placeholder (populate later) ---
ALIASES = {
        '6 ave': 'ave of the americas',
    'avenue of the americas': 'ave of the americas',
    'west 110 street': '110 st',
    'andrews avenue north': 'andrews avenue', 'andrews avenue south': 'andrews avenue'
}

# --- Normalization function ---
def normalize(name: str) -> str:
    """Canonicalize street name with lowercase, punctuation removal,
       abbreviation/ordinal expansion, and alias replacement."""
    if not isinstance(name, str):
        return ''
    s = name.lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    words = []
    for w in s.split():
        if w in ORDINAL:
            w = ORDINAL[w]
        elif w in ABBREV:
            w = ABBREV[w]
        words.append(w)
    s_norm = ' '.join(words)
    # Apply alias mapping if exists
    s_norm = ALIASES.get(s_norm, s_norm)
    return s_norm

# --- Preprocess DOT dataset ---
dot_prep = (
    dot[['Borough Code', 'Location On Street']]
    .dropna()
    .assign(**{
        'Borough Code Num': lambda df: df['Borough Code'].map(borough_dct),
        'Location On Street': lambda df: df['Location On Street'].str.lower()
    })
    .drop_duplicates(['Borough Code Num', 'Location On Street'])
)
dot_prep['norm'] = dot_prep['Location On Street'].map(normalize)

# --- Preprocess CSCL dataset ---
cscl_prep = cscl.copy()
cscl_prep['Borough Code'] = cscl_prep['Borough Code'].astype(str)
cscl_prep['Full Street Name'] = cscl_prep['Full Street Name'].str.lower()
cscl_prep['norm'] = cscl_prep['Full Street Name'].map(normalize)

# --- Merge on borough + normalized name ---
# --- Merge on borough + normalized name, include PHYSICALID ---
merged = dot_prep.merge(
    cscl_prep[['Borough Code', 'Full Street Name', 'PHYSICALID', 'norm']],
    left_on=['Borough Code Num', 'norm'],
    right_on=['Borough Code', 'norm'],
    how='left',
    suffixes=('_dot', '_cscl')
)


print(merged.columns.tolist())



['Borough Code_dot', 'Location On Street', 'Borough Code Num', 'norm', 'Borough Code_cscl', 'Full Street Name', 'PHYSICALID']


In [167]:
# --- Result DataFrame ---
# --- Select and rename columns for final result ---
result = merged[['Borough Code Num', 'Location On Street', 'Full Street Name', 'PHYSICALID']].rename(
    columns={
        'Borough Code Num': 'Borough Code',
        'Location On Street': 'DOT street name',
        'Full Street Name': 'CSCL street name'
    }
).drop_duplicates()

result['PHYSICALID'] = result['PHYSICALID'].apply(lambda x: int(x) if pd.notna(x) else x)

# --- Optional: mapping dictionary ---
mapping = (
    merged.groupby(['Borough Code Num', 'Location On Street'])['Full Street Name']
    .apply(lambda x: set(x.dropna()))
    .to_dict()
)


In [ ]:
result

In [106]:
result.columns

Index(['Borough Code', 'DOT street name', 'CSCL street name', 'PHYSICALID'], dtype='object')

In [ ]:
# remove Missing value (always substring)
# check that mapping is 1-to-1
# see condensed list of map after this verification

In [168]:
missing_cscl = result[result['CSCL street name'].isna()]

In [169]:
missing_cscl

,Borough Code,DOT street name,CSCL street name,PHYSICALID
696,5,bend,NaN,NaN
924,1,6 avenue,NaN,NaN
1335,1,miller hy et nb,NaN,NaN
1336,1,miller hy en sb,NaN,NaN
2370,1,bend,NaN,NaN
...,...,...,...,...
101324,4,bklyn qns expressway exit sb,NaN,NaN
101415,3,columbia street esplanade,NaN,NaN
101420,3,st mark's avenue,NaN,NaN
101470,3,gowanus expressway exit 24 wb,NaN,NaN


In [177]:
# look at what corresponds to given DOT name

dot_name = "beach"

# Lowercase and split into words
dot_words = dot_name.lower().split()

# Filter CSCL rows by borough first, if desired
cscl_candidates = cscl_prep[cscl_prep['Borough Code'] == '2']  # replace '1' with borough code

# Check if all words appear as substrings in CSCL street
def contains_all_words(cscl_street, words):
    cscl_street_lower = cscl_street.lower()
    return all(w in cscl_street_lower for w in words)

matches = cscl_candidates[cscl_candidates['Full Street Name'].apply(lambda x: contains_all_words(x, dot_words))]

matches[['Full Street Name', 'PHYSICALID']]

,Full Street Name,PHYSICALID
902,orchard beach rd,71106
1595,beach ave,41315
2400,beach ave,164343
2559,beach ave pedestrian opas,177133
5298,orchard beach rd,71103
8248,beach ave,57577
12689,orchard beach rd,71100
14493,beach ave pedestrian opas,177134
21738,beach ave,41316
22019,beach ave,57581


In [154]:
s = cscl[cscl["PHYSICALID"] == 87765]["Full Street Name"].values[0]
s.lower() == 'andrews ave'
normalize('andrews ave')

s = dot[dot["Location On Street"] == "ANDREWS AVENUE"]["Location On Street"].values[0]
s.split(' ')

['ANDREWS', 'AVENUE']

In [ ]:
# manually assign CSCL street name to given DOT street name. PHYSICALID will be assigned randomly based 
# on existing CSCL street name columns

manual_mapping = {
    '6 ave': 'ave of the americas',
    'avenue of the americas': 'ave of the americas',
    'west 110 street': '110 st',
    'andrews avenue north': 'andrews avenue',
    'andrews avenue south': 'andrews avenue'
}

# Filter rows that need manual fill
for dot_street, cscl_street in manual_mapping.items():
    # Get candidate PHYSICALIDs from CSCL
    candidates = cscl_prep[cscl_prep['Full Street Name'] == cscl_street]['PHYSICALID']
    
    if len(candidates) == 0:
        print(f"No CSCL entry found for {cscl_street}")
        continue
    
    chosen_physicalid = candidates.sample(1).iloc[0]  # randomly pick one PHYSICALID
    
    # Update the result df
    result.loc[result['DOT street name'] == dot_street, 'CSCL street name'] = cscl_street
    result.loc[result['DOT street name'] == dot_street, 'PHYSICALID'] = chosen_physicalid


In [ ]:
# remove rows from result with junk DOT names

to_remove = ['bend', 'dead end', 'edens alley', 'mc kenna square', 'ramp', 'dr m l king jr boulevard', 'bissel avenue']
result = result[~result['DOT street name'].isin(to_remove)]